Import some necessary libraries

In [ ]:
import csv
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import sys

In [ ]:
# prompt: dapatkan versi dari tensorflow yang digunakan

tf.__version__


In [ ]:
print(f"Python version: {sys.version}")
print(f"Python version info: {sys.version_info}")

Get a list of product names and their labels from the csv file of the previous notebook dataset

In [ ]:
def parse_data_from_file():
  product_names = []
  labels = []

  with open('shuffled_data.csv', mode='r') as file:
    reader = csv.reader(file)
    next(reader)

    for row in reader:
      product_names.append(row[0])
      labels.append(row[1])

  return product_names, labels

In [ ]:
product_names, labels = parse_data_from_file()

print(f'length of product names: {len(product_names)}')
print(f'first product name: {product_names[0]}')
print(f'first product name has {len(product_names[0].split())} words')
print(f'length of labels: {len(labels)}')
print(f'daftar labels adalah {list(set(labels))}')

Function to get training and validation datasets, then display the amount of data

In [ ]:
def train_val_datasets():
  train_size = int(len(product_names) * 0.9)

  train_product_names = product_names[:train_size]
  train_labels = labels[:train_size]

  val_product_names = product_names[train_size:]
  val_labels = labels[train_size:]

  train_dataset = tf.data.Dataset.from_tensor_slices((train_product_names, train_labels))
  val_dataset = tf.data.Dataset.from_tensor_slices((val_product_names, val_labels))

  return train_dataset, val_dataset

In [ ]:
train_dataset, validation_dataset = train_val_datasets()

print(f"there are {train_dataset.cardinality()} product name - label pairs for training.\n")
print(f"there are {validation_dataset.cardinality()} product name - label pairs for validation.\n")

Function definition to standardize product name

In [ ]:
@tf.keras.utils.register_keras_serializable()
def standardize_product_name(product_name):
  product_name = tf.strings.lower(product_name)
  product_name = tf.strings.regex_replace(
      product_name, r"\b\d+(\.\d+)?x\d+(\.\d+)?(g|ml|kg|lt)\b", ""
  )
  product_name = tf.strings.regex_replace(
      product_name, r"\b\d+(\.\d+)?(g|ml|kg|lt)\b", ""
  )
  product_name = tf.strings.regex_replace(product_name, r"\b\d+(g|ml|kg|lt)\b", "")
  product_name = tf.strings.regex_replace(product_name, r"\b\d+\'s\b", "")
  product_name = tf.strings.regex_replace(product_name, r"[^a-z\s]", "")
  product_name = tf.strings.regex_replace(product_name, r'\s{2,}', ' ')
  product_name = tf.strings.strip(product_name)

  return product_name

In [ ]:
import random

random_product_names = random.sample(product_names, 10)
for random_product_name in random_product_names:
  standardized_product_name = standardize_product_name(random_product_name)

  print(f'product name: {random_product_name}')
  print(f'standardized product name: {standardized_product_name}\n')

In [ ]:
VOCAB_SIZE = 5000
EMBEDDING_DIM = 16
MAX_LENGTH = 120

Define a vectorizer based on some previously defined variables

In [ ]:
def fit_vectorizer(product_names):
  vectorizer = tf.keras.layers.TextVectorization(
      standardize=standardize_product_name,
      output_sequence_length=MAX_LENGTH,
      max_tokens=VOCAB_SIZE,
  )
  vectorizer.adapt(product_names)

  return vectorizer

In [ ]:
vectorizer = fit_vectorizer(train_dataset.map(lambda product_name, label: product_name))
vocab_size = vectorizer.vocabulary_size()

print(f"vocabulary contains {vocab_size} words\n")

Define encoder based on training and validation labels

In [ ]:
def fit_encoder():
  train_labels = train_dataset.map(lambda product_name, label: label)
  validation_labels = validation_dataset.map(lambda product_name, label: label)
  labels = train_labels.concatenate(validation_labels)

  encoder = tf.keras.layers.StringLookup(
      # oov_token='<UNK>',
      num_oov_indices=0,
  )
  encoder.adapt(labels)

  return encoder

In [ ]:
encoder = fit_encoder()

print(f'unique labels: {encoder.get_vocabulary()}')

Preprocess the training and validation datasets and check their cardinality.

In [ ]:
def preprocess_dataset(dataset, vectorizer, encoder):
  dataset = dataset.map(lambda product_name, label: (vectorizer(product_name), encoder(label)))
  dataset = dataset.batch(32)

  return dataset

In [ ]:
train_proc_dataset = preprocess_dataset(train_dataset, vectorizer, encoder)
validation_proc_dataset = preprocess_dataset(validation_dataset, vectorizer, encoder)

print(f"number of batches in the train dataset: {train_proc_dataset.cardinality()}")
print(f"number of batches in the validation dataset: {validation_proc_dataset.cardinality()}")

In [ ]:
train_batch = next(train_proc_dataset.as_numpy_iterator())
validation_batch = next(validation_proc_dataset.as_numpy_iterator())

print(f"shape of the train batch: {train_batch[0].shape}")
print(f"shape of the validation batch: {validation_batch[0].shape}")

Define NLP model with 6 categories

In [ ]:
def create_model():
  model = tf.keras.Sequential([
    tf.keras.Input(shape=(MAX_LENGTH,)),
    tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(6, activation='softmax')
  ])

  model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
  )

  return model

Get a summary of the NLP model

In [ ]:
model = create_model()
model.summary()

In [ ]:
example_batch = train_proc_dataset.take(1)

try:
  model.evaluate(example_batch, verbose=False)
except:
	print("your model is not compatible with the dataset you defined earlier. check that the loss function and last layer are compatible with one another.")
else:
	predictions = model.predict(example_batch, verbose=False)
	print(f"predictions have shape: {predictions.shape}")

Train the NLP model with 50 epochs

In [ ]:
history = model.fit(
    train_proc_dataset,
    epochs=50,
    validation_data=validation_proc_dataset,
)

Display the accuracy and loss results using a plot

In [ ]:
def plot_graphs(history, metric):
  plt.plot(history.history[metric])
  plt.plot(history.history[f'val_{metric}'])
  plt.xlabel("epochs")
  plt.ylabel(metric)
  plt.legend([metric, f'val_{metric}'])
  plt.show()

plot_graphs(history, "accuracy")
plot_graphs(history, "loss")

Export the NLP, vectorizer, and encoder models using pickle so they can be used in the ML API.

In [ ]:
# export model
import pickle

if not os.path.exists(os.path.join('model')):
  os.mkdir(os.path.join('model'))

model.save(os.path.join('model', 'model.keras'))

with open(os.path.join('model', 'vectorizer.pickle'), 'wb') as f:
  pickle.dump(vectorizer, f)

with open(os.path.join('model', 'encoder.pickle'), 'wb') as f:
  pickle.dump(encoder, f)

In [ ]:
!zip -r /content/product-classification-model.zip /content/model